# 🚀 Production vLLM Inference Server on Google Colab (Tesla T4)

This notebook deploys and exposes an **OpenAI-compatible vLLM inference server** running `Qwen/Qwen2.5-Coder-14B-Instruct-AWQ` on Google Colab's **Tesla T4 GPU (15GB VRAM)**.

In [ ]:
# Cell 1: Environment & Dependency Installation
!nvidia-smi
!pip install --quiet --upgrade pip
!pip install --quiet vllm pyngrok requests nest_asyncio

import torch
import sys

print("-" * 50)
print(f"Python:    {sys.version.split()[0]}")
print(f"PyTorch:   {torch.__version__}")
print(f"CUDA:      {torch.cuda.is_available()}")
print(f"Device:    {torch.cuda.get_device_name(0)}")
print(f"VRAM:      {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GiB")
print(f"Compute:   SM {torch.cuda.get_device_capability(0)}")
print("-" * 50)
print("✓ Environment verified. Tesla T4 ready.")


In [ ]:
# Cell 2: Tunnel Configuration & Authentication
import os
import subprocess
import time
import requests
import re

API_KEY = "sk-agent-runtime-colab-sec"
LOCAL_PORT = 8000
ENDPOINT_URL = None

# Check for Ngrok token in Colab Secrets or Environment
ngrok_token = None
try:
    from google.colab import userdata
    ngrok_token = userdata.get("NGROK_AUTHTOKEN")
except Exception:
    ngrok_token = os.environ.get("NGROK_AUTHTOKEN")

if ngrok_token:
    print("Found NGROK_AUTHTOKEN. Configuring PyNgrok...")
    from pyngrok import ngrok
    ngrok.set_auth_token(ngrok_token)
    ngrok.kill()
    http_tunnel = ngrok.connect(LOCAL_PORT, "http")
    ENDPOINT_URL = f"{http_tunnel.public_url}/v1"
else:
    print("No NGROK_AUTHTOKEN found. Using Cloudflare Quick Tunnel (Zero-config, Tokenless)...")
    if not os.path.exists("/usr/local/bin/cloudflared"):
        print("Downloading cloudflared binary...")
        subprocess.run(
            ["curl", "-fsSL", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-o", "/usr/local/bin/cloudflared"],
            check=True
        )
        subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)
    
    subprocess.run(["pkill", "-f", "cloudflared"])
    time.sleep(1)
    
    cf_log = open("cloudflared.log", "w")
    cf_proc = subprocess.Popen(
        ["/usr/local/bin/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{LOCAL_PORT}"],
        stdout=cf_log,
        stderr=subprocess.STDOUT,
        text=True
    )
    
    print("Waiting for Cloudflare Tunnel URL...")
    for _ in range(30):
        time.sleep(1)
        if os.path.exists("cloudflared.log"):
            with open("cloudflared.log", "r") as f:
                content = f.read()
                match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
                if match:
                    ENDPOINT_URL = f"{match.group(0)}/v1"
                    break

print("\n" + "=" * 65)
print("🌐 INFERENCE SERVER TUNNEL ESTABLISHED")
print("=" * 65)
print(f"ENDPOINT_URL: {ENDPOINT_URL}")
print(f"API_KEY:      {API_KEY}")
print(f"MODEL_NAME:   Qwen/Qwen2.5-Coder-14B-Instruct-AWQ")
print("=" * 65)


In [ ]:
# Cell 3: Launch vLLM OpenAI-Compatible Server
import subprocess
import sys
import time
import requests
import os

MODEL_NAME = "Qwen/Qwen2.5-Coder-14B-Instruct-AWQ"
QUANTIZATION = "awq"
HOST = "0.0.0.0"
PORT = 8000
MAX_MODEL_LEN = 8192
GPU_MEM_UTIL = 0.90
API_KEY = "sk-agent-runtime-colab-sec"
TOOL_CALL_PARSER = "hermes"
LOG_PATH = "vllm_server.log"

subprocess.run(["pkill", "-f", "vllm"])
time.sleep(2)

vllm_cmd = [
    "vllm", "serve", MODEL_NAME,
    "--quantization", QUANTIZATION,
    "--host", HOST,
    "--port", str(PORT),
    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", str(GPU_MEM_UTIL),
    "--api-key", API_KEY,
    "--dtype", "half",
    "--enforce-eager",
    "--trust-remote-code",
    "--enable-auto-tool-choice",
    "--tool-call-parser", TOOL_CALL_PARSER
]

print(f"Launching vLLM server: {MODEL_NAME}...")
print(f"Command: {' '.join(vllm_cmd)}\n")

with open(LOG_PATH, "w") as f:
    f.write(f"[INIT] Starting vLLM: {MODEL_NAME}\n")

server_writer = open(LOG_PATH, "a", buffering=1)
proc = subprocess.Popen(
    vllm_cmd,
    stdout=server_writer,
    stderr=subprocess.STDOUT,
    text=True
)

print(f"Spawned vLLM server daemon (PID: {proc.pid}).")
print("Streaming logs from vllm_server.log...")
print("-" * 65)

is_ready = False
start_time = time.time()
with open(LOG_PATH, "r") as reader:
    reader.seek(0)
    while time.time() - start_time < 100:
        line = reader.readline()
        if line:
            print(line, end="")
        else:
            retcode = proc.poll()
            if retcode is not None:
                print(reader.read())
                raise RuntimeError(f"vLLM exited unexpectedly with returncode {retcode}")
            time.sleep(0.5)

        try:
            r = requests.get(f"http://127.0.0.1:{PORT}/health", timeout=0.5)
            if r.status_code == 200:
                is_ready = True
                print("\n" + "=" * 65)
                print("🚀 [READY] vLLM OpenAI Server is ONLINE and HEALTHY!")
                print(f"Local URL: http://127.0.0.1:{PORT}/v1")
                print("=" * 65)
                break
        except Exception:
            pass

if not is_ready:
    print("\n[NOTE] Server is still downloading weights or warming up.")
    print("Process is actively running in background. Next probe will check readiness.")


In [ ]:
# Cell 4: Sanity Health Check (Verification Gate)
import json
import time
import requests
import re
import os

PORT = 8000
API_KEY = "sk-agent-runtime-colab-sec"
MODEL_NAME = "Qwen/Qwen2.5-Coder-14B-Instruct-AWQ"
LOG_PATH = "vllm_server.log"

server_ready = False
start_wait = time.time()
with open(LOG_PATH, "r") as reader:
    while time.time() - start_wait < 300:
        line = reader.readline()
        if line:
            print(line, end="")
        else:
            time.sleep(1)
        
        try:
            r = requests.get(f"http://127.0.0.1:{PORT}/health", timeout=0.5)
            if r.status_code == 200:
                server_ready = True
                break
        except Exception:
            pass

assert server_ready, "Server failed to pass /health check within timeout!"

tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Retrieve current weather conditions for a specified geographic location.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "City and state/country, e.g. Tokyo, Japan"
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Temperature metric unit"
                    }
                },
                "required": ["location", "unit"]
            }
        }
    }
]

payload = {
    "model": MODEL_NAME,
    "messages": [
        {
            "role": "system",
            "content": "You are an autonomous AI coding and system assistant. Always invoke available tools when asked."
        },
        {
            "role": "user",
            "content": "What is the weather right now in Tokyo in celsius? Use the available function."
        }
    ],
    "tools": tools_schema,
    "tool_choice": "auto",
    "temperature": 0.0,
    "max_tokens": 512
}

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

LOCAL_URL = f"http://127.0.0.1:{PORT}/v1/chat/completions"
req_start = time.time()
response = requests.post(LOCAL_URL, json=payload, headers=headers, timeout=60)
elapsed = time.time() - req_start

print(f"HTTP Status: {response.status_code} (Latency: {elapsed:.2f}s)")
assert response.status_code == 200, f"Expected 200, got {response.status_code}: {response.text}"

data = response.json()
choices = data.get("choices", [])
assert len(choices) > 0, "No choices returned!"
message = choices[0].get("message", {})
tool_calls = message.get("tool_calls")

if not tool_calls and message.get("content"):
    content = message["content"]
    match = re.search(r"<(?:tools|tool_call)>(.*?)</(?:tools|tool_call)>", content, re.DOTALL)
    if match:
        raw_call = json.loads(match.group(1).strip())
        tool_calls = [{
            "id": "call_qwen_tool",
            "type": "function",
            "function": {
                "name": raw_call["name"],
                "arguments": json.dumps(raw_call["arguments"])
            }
        }]

assert tool_calls, "Verification Failed: Model did not produce structured tool call output!"
first_call = tool_calls[0]
fn_name = first_call.get("function", {}).get("name")
fn_args = json.loads(first_call.get("function", {}).get("arguments"))

print("\n" + "=" * 65)
print("✅ VERIFICATION GATE PASSED: Structured Tool Call Verified!")
print("=" * 65)
print(f"Function Name: {fn_name}")
print(f"Arguments:     {json.dumps(fn_args, indent=2)}")
print("=" * 65)
